# Sentence Corpus Construction

## Overview
This notebook builds a sentence-level corpus from validated source URLs for health equity analysis. It reads source metadata, retrieves content from PDF and HTML documents, extracts clean text, and generates analysis-ready sentence outputs.

The objective of this notebook is to produce a high-quality, traceable text corpus with clear quality-control artifacts before downstream modeling and inference.

## Objectives
- Inspect and load validated source records
- Fetch source content reliably with retry and timeout safeguards
- Extract text from PDF and HTML sources using consistent parsing logic
- Segment cleaned text into sentence-level observations
- Generate corpus outputs for full and equity-focused analyses
- Produce quality-control, exclusion, and failure logs for auditability

## Dataset Description
The workflow starts from `validated_dataset.csv`, where each row represents a source document linked to a project record. The notebook processes each source, applies extraction rules, and writes sentence-level outputs plus supporting logs in the working directory.

Core outputs include `full_sentence_corpus.csv`, `equity_sentence_subset.csv`, `extraction_qc_report.csv`, `exclusion_log.csv`, and `failure_log.csv`.

## Key Considerations
- Source links may fail intermittently due to network or host constraints
- PDF/HTML format differences can affect extraction consistency
- Very short, very long, or noisy sentences can reduce downstream quality
- Some documents may be excluded due to page limits or parse failures
- Reproducible document and sentence identifiers are required for traceability

## Outcome
The resulting corpus and logs provide a transparent foundation for subsequent text analysis, feature engineering, and model development.

# Sentence-Level Corpus Builder — Health Equity Text Analysis
### IDS 570: Text as Data — Final Project

This notebook reads `validated_dataset.csv`, fetches every source URL, extracts and cleans text, segments it into sentences with spaCy, and writes all outputs to **../01_data/raw_data/**.

**Outputs produced (written to ../01_data/raw_data/):**
- `full_sentence_corpus.csv`
- `equity_sentence_subset.csv`
- `extraction_qc_report.csv`
- `exclusion_log.csv`
- `failure_log.csv`
- `corpus_build.log`


In [1]:
# ── Install dependencies (run once) ──────────────────────────
!pip install pandas requests beautifulsoup4 pdfplumber spacy -q
!python -m spacy download en_core_web_sm -q



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [ ]:
# ── Imports & configuration ──────────────────────────────────
import csv, hashlib, io, logging, os, re, sys, time, traceback
from collections import Counter
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
import spacy
from bs4 import BeautifulSoup

try:
    import pdfplumber

    PDF_BACKEND = "pdfplumber"
except ImportError:
    from pypdf import PdfReader

    PDF_BACKEND = "pypdf"

# ── All outputs go to 01_data/raw_data folder ────────────────
OUTPUT_DIR = Path("../01_data/raw_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Page-count filter thresholds ─────────────────────────────
MIN_PAGES = 20
MAX_PAGES = 150

# ── Request settings ─────────────────────────────────────────
REQUEST_TIMEOUT = 60
RETRY_ATTEMPTS = 2
RETRY_DELAY = 5

# ── Sentence quality thresholds ──────────────────────────────
MIN_SENTENCE_CHARS = 20
MAX_SENTENCE_CHARS = 3000

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(
            OUTPUT_DIR / "corpus_build.log", mode="w", encoding="utf-8"
        ),
    ],
)
log = logging.getLogger("corpus_builder")

print(f"PDF backend : {PDF_BACKEND}")
print(f"Output dir  : {OUTPUT_DIR.resolve()}")

PDF backend : pdfplumber
Output dir  : /Users/far/Downloads/TEXTasDATA


In [3]:
# ── Load spaCy model ─────────────────────────────────────────
NLP = spacy.load("en_core_web_sm")
NLP.max_length = 5_000_000
print(f"spaCy model loaded: {NLP.meta['name']} v{NLP.meta['version']}")

spaCy model loaded: core_web_sm v3.8.0


## Helper functions

In [4]:
# ── Helpers ──────────────────────────────────────────────────


def make_document_id(url: str) -> str:
    """Deterministic short hash — same URL always gets the same id."""
    return "doc_" + hashlib.sha256(url.encode()).hexdigest()[:12]


def detect_source_type(url: str) -> str:
    """Heuristic: PDF vs HTML based on URL extension."""
    lower = url.lower().split("?")[0].split("#")[0]
    return "pdf" if lower.endswith(".pdf") else "html"


def fetch_content(url: str) -> requests.Response:
    """GET with retries, timeout, and realistic User-Agent."""
    headers = {"User-Agent": USER_AGENT}
    for attempt in range(1, RETRY_ATTEMPTS + 1):
        try:
            resp = requests.get(
                url, headers=headers, timeout=REQUEST_TIMEOUT, allow_redirects=True
            )
            resp.raise_for_status()
            return resp
        except Exception as exc:
            log.warning(
                "Attempt %d/%d failed for %s: %s", attempt, RETRY_ATTEMPTS, url, exc
            )
            if attempt < RETRY_ATTEMPTS:
                time.sleep(RETRY_DELAY)
    raise RuntimeError(f"All {RETRY_ATTEMPTS} attempts failed for {url}")


print("Helpers defined ✓")

Helpers defined ✓


## PDF extraction

In [5]:
# ── PDF extraction ───────────────────────────────────────────


def get_pdf_page_count(content_bytes: bytes) -> Optional[int]:
    """Return number of pages, or None if unreadable."""
    try:
        if PDF_BACKEND == "pdfplumber":
            with pdfplumber.open(io.BytesIO(content_bytes)) as pdf:
                return len(pdf.pages)
        else:
            reader = PdfReader(io.BytesIO(content_bytes))
            return len(reader.pages)
    except Exception:
        return None


def extract_pdf_text(content_bytes: bytes) -> str:
    """Extract readable text from all pages of a PDF."""
    pages_text = []
    try:
        if PDF_BACKEND == "pdfplumber":
            with pdfplumber.open(io.BytesIO(content_bytes)) as pdf:
                for page in pdf.pages:
                    txt = page.extract_text() or ""
                    pages_text.append(txt)
        else:
            reader = PdfReader(io.BytesIO(content_bytes))
            for page in reader.pages:
                txt = page.extract_text() or ""
                pages_text.append(txt)
    except Exception as exc:
        log.error("PDF text extraction error: %s", exc)
    return "\n".join(pages_text)


print("PDF extraction defined ✓")

PDF extraction defined ✓


## HTML extraction (with PubMed abstract handling)

In [6]:
# ── HTML extraction ──────────────────────────────────────────


def extract_html_text(html: str, url: str) -> str:
    """
    Extract main-body text from an HTML page.
    PubMed pages → abstract only.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Remove boilerplate tags
    for tag in soup.find_all(
        [
            "nav",
            "header",
            "footer",
            "aside",
            "script",
            "style",
            "noscript",
            "iframe",
            "form",
            "button",
            "svg",
        ]
    ):
        tag.decompose()

    # Remove boilerplate by class / id patterns
    bp = re.compile(
        r"(cookie|consent|banner|sidebar|menu|nav|footer|advertisement|"
        r"popup|modal|social[-_]?share|related[-_]?articles|comment|"
        r"newsletter|signup)",
        re.I,
    )
    for el in soup.find_all(True, {"class": bp}):
        el.decompose()
    for el in soup.find_all(True, {"id": bp}):
        el.decompose()

    # ── PubMed special handling ──
    is_pubmed = "pubmed.ncbi.nlm.nih.gov" in url or "ncbi.nlm.nih.gov/pubmed" in url
    if is_pubmed:
        return _extract_pubmed_abstract(soup)

    # ── Generic: prefer <main> or <article> ──
    main = soup.find("main") or soup.find("article")
    if main:
        return main.get_text(separator="\n", strip=True)

    body = soup.find("body")
    if body:
        return body.get_text(separator="\n", strip=True)
    return soup.get_text(separator="\n", strip=True)


def _extract_pubmed_abstract(soup: BeautifulSoup) -> str:
    """Extract ONLY the abstract from a PubMed page."""
    abstract_div = soup.find("div", class_=re.compile(r"abstract", re.I))
    if abstract_div:
        return abstract_div.get_text(separator="\n", strip=True)

    abstract_tag = soup.find("abstract")
    if abstract_tag:
        return abstract_tag.get_text(separator="\n", strip=True)

    for heading in soup.find_all(["h2", "h3", "h4"]):
        if "abstract" in heading.get_text(strip=True).lower():
            parts = []
            for sib in heading.find_next_siblings():
                if sib.name and sib.name.startswith("h"):
                    break
                parts.append(sib.get_text(separator=" ", strip=True))
            if parts:
                return "\n".join(parts)
    return ""


print("HTML extraction defined ✓")

HTML extraction defined ✓


## Text cleaning

In [7]:
# ── Text cleaning ────────────────────────────────────────────

_RE_PAGE_NUMBERS = re.compile(r"^\s*\d{1,4}\s*$", re.MULTILINE)
_RE_DOT_LEADERS = re.compile(r"\.{4,}")
_RE_OCR_SPACED = re.compile(r"(?:[A-Z]\s){3,}[A-Z]")
_RE_MULTI_SPACE = re.compile(r"[ \t]{2,}")
_RE_MULTI_NEWLINE = re.compile(r"\n{3,}")
_RE_HEADER_FOOTER = re.compile(
    r"(?i)^.{0,5}(page\s+\d+|www\.|http|©|all rights reserved|"
    r"table of contents|acknowledgments?).{0,80}$",
    re.MULTILINE,
)


def clean_text(raw: str) -> str:
    """Normalize and clean extracted text."""
    if not raw:
        return ""
    text = raw

    # Fix encoding artifacts
    for old, new in [
        ("\u00a0", " "),
        ("\u2019", "'"),
        ("\u2018", "'"),
        ("\u201c", '"'),
        ("\u201d", '"'),
        ("\u2013", "–"),
        ("\u2014", "—"),
        ("\uf0b7", ""),
        ("\x0c", "\n"),
    ]:
        text = text.replace(old, new)

    # Remove noise patterns
    text = _RE_PAGE_NUMBERS.sub("", text)
    text = _RE_DOT_LEADERS.sub("", text)
    text = _RE_OCR_SPACED.sub("", text)
    text = _RE_HEADER_FOOTER.sub("", text)

    # Normalize whitespace
    text = _RE_MULTI_SPACE.sub(" ", text)
    text = _RE_MULTI_NEWLINE.sub("\n\n", text)

    # Remove duplicate lines (keep first occurrence)
    seen = set()
    deduped = []
    for line in text.split("\n"):
        stripped = line.strip()
        if stripped and stripped not in seen:
            seen.add(stripped)
            deduped.append(stripped)
        elif not stripped:
            deduped.append("")
    return "\n".join(deduped).strip()


print("Text cleaning defined ✓")

Text cleaning defined ✓


## Sentence segmentation & quality filter

In [8]:
# ── Sentence segmentation & filtering ────────────────────────

_RE_ALL_CAPS_LINE = re.compile(r"^[A-Z\s\d\W]{5,}$")
_RE_TOC_LINE = re.compile(r"^\d+[\.\)]\s")
_RE_HEADING_LIKE = re.compile(r"^(chapter|section|appendix|figure|table)\s", re.I)


def is_valid_sentence(text: str) -> bool:
    """Return True if text looks like a real English sentence."""
    t = text.strip()
    if len(t) < MIN_SENTENCE_CHARS or len(t) > MAX_SENTENCE_CHARS:
        return False
    if _RE_ALL_CAPS_LINE.match(t):
        return False
    if _RE_TOC_LINE.match(t):
        return False
    if _RE_HEADING_LIKE.match(t):
        return False
    if not re.search(r"[a-z]", t):
        return False
    words = [w for w in t.split() if re.search(r"[a-zA-Z]", w)]
    if len(words) < 3:
        return False
    return True


def segment_sentences(cleaned: str) -> list:
    """Use spaCy to split text into sentences, then filter."""
    if not cleaned:
        return []
    doc = NLP(cleaned)
    sentences = []
    for sent in doc.sents:
        s = sent.text.strip()
        s = re.sub(r"\s*\n\s*", " ", s)
        s = re.sub(r"\s{2,}", " ", s)
        if is_valid_sentence(s):
            sentences.append(s)
    return sentences


print("Sentence segmentation defined ✓")

Sentence segmentation defined ✓


## Feature columns (equity flags, related terms, definitions)

In [9]:
# ── Feature columns ──────────────────────────────────────────

_RE_EQUITY = re.compile(r"\bequity\b", re.I)
_RE_EQUITIES = re.compile(r"\bequities\b", re.I)
_RE_HEALTH_EQUITY = re.compile(r"\bhealth\s+equity\b", re.I)

_RELATED_TERMS = re.compile(
    r"\b(disparit(?:y|ies)|inequalit(?:y|ies)|access|underserved|"
    r"social\s+determinants?|sdoh)\b",
    re.I,
)

_DEFINITION_PATTERNS = [
    re.compile(r"\b(?:is|are)\s+defined\s+as\b", re.I),
    re.compile(r"\b(?:refers?\s+to|meaning|means)\b", re.I),
    re.compile(r"\bequity\s+(?:is|means|refers)\b", re.I),
    re.compile(r"\bdefine[sd]?\s+(?:health\s+)?equity\b", re.I),
    re.compile(r"\bhealth\s+equity\s+is\b", re.I),
    re.compile(r"\bwe\s+define\b", re.I),
]


def compute_features(sentence: str) -> dict:
    """Compute all per-sentence feature columns."""
    contains_equity = bool(_RE_EQUITY.search(sentence))
    contains_equities = bool(_RE_EQUITIES.search(sentence))
    contains_health_equity = bool(_RE_HEALTH_EQUITY.search(sentence))
    contains_related = bool(_RELATED_TERMS.search(sentence))
    is_def = any(p.search(sentence) for p in _DEFINITION_PATTERNS)
    tokens = sentence.split()

    return {
        "contains_equity": contains_equity,
        "contains_equities": contains_equities,
        "contains_health_equity": contains_health_equity,
        "contains_related_terms": contains_related,
        "is_definition_sentence": is_def,
        "word_count": len(tokens),
        "char_count": len(sentence),
        "sentence_length_tokens": len(tokens),
    }


print("Feature columns defined ✓")

Feature columns defined ✓


## Document subtype classification (rule-based)

In [10]:
# ── Document subtype heuristic ────────────────────────────────


def classify_subtype(title: str, category: str, text_sample: str) -> str:
    """
    Rule-based classification:
      conceptual | empirical | policy_report | commentary
    """
    t = (title + " " + text_sample[:2000]).lower()

    if category == "news_commentary":
        return "commentary"
    if category in ("federal_policy", "state_local"):
        return "policy_report"

    empirical_kw = [
        "method",
        "result",
        "findings",
        "sample",
        "regression",
        "survey",
        "cohort",
        "trial",
        "odds ratio",
        "p-value",
        "participants",
        "cross-sectional",
        "longitudinal",
        "data were",
        "data was",
        "n =",
        "n=",
    ]
    conceptual_kw = [
        "concept",
        "framework",
        "theory",
        "definition",
        "conceptual",
        "normative",
        "discourse",
        "philosophical",
        "scoping review",
        "narrative review",
        "meta-narrative",
    ]
    commentary_kw = [
        "editorial",
        "commentary",
        "perspective",
        "opinion",
        "viewpoint",
        "letter to",
    ]

    scores = {
        "empirical": sum(1 for kw in empirical_kw if kw in t),
        "conceptual": sum(1 for kw in conceptual_kw if kw in t),
        "commentary": sum(1 for kw in commentary_kw if kw in t),
    }
    best = max(scores, key=scores.get)
    if scores[best] == 0:
        return "policy_report" if category == "ngo_nonprofit" else "conceptual"
    return best


print("Subtype classifier defined ✓")

Subtype classifier defined ✓


## Single-source processing function

In [11]:
# ── Process one source ────────────────────────────────────────


def process_one_source(row: dict) -> dict:
    """
    Fetch → extract → clean → segment one source.
    Returns dict with status, page_count, sentences, etc.
    """
    url = row["url"]
    title = row["title"]
    category = row["category"]
    source_type = detect_source_type(url)

    result = {
        "url": url,
        "title": title,
        "category": category,
        "source_label": row.get("source_label", ""),
        "source_type": source_type,
        "page_count": None,
        "sentences": [],
        "exclusion_reason": None,
        "status": "success",
        "notes": "",
        "raw_text_sample": "",
    }

    # ── Fetch ──
    try:
        resp = fetch_content(url)
    except Exception as exc:
        result["status"] = "fetch_failed"
        result["notes"] = str(exc)
        return result

    content_type = resp.headers.get("Content-Type", "").lower()
    if "application/pdf" in content_type:
        source_type = "pdf"
        result["source_type"] = "pdf"

    # ── PDF path ──
    if source_type == "pdf":
        content_bytes = resp.content
        page_count = get_pdf_page_count(content_bytes)
        result["page_count"] = page_count

        if page_count is None:
            result["status"] = "failed"
            result["notes"] = "Could not determine PDF page count"
            return result

        if page_count < MIN_PAGES:
            result["status"] = "excluded"
            result["exclusion_reason"] = f"page_count={page_count} < {MIN_PAGES}"
            return result
        if page_count > MAX_PAGES:
            result["status"] = "excluded"
            result["exclusion_reason"] = f"page_count={page_count} > {MAX_PAGES}"
            return result

        raw_text = extract_pdf_text(content_bytes)

    # ── HTML path ──
    else:
        raw_text = extract_html_text(resp.text, url)
        result["page_count"] = None

    if not raw_text or len(raw_text.strip()) < 50:
        result["status"] = "failed"
        result["notes"] = "No meaningful text extracted"
        return result

    # ── Clean → segment ──
    cleaned = clean_text(raw_text)
    result["raw_text_sample"] = cleaned[:500]
    result["sentences"] = segment_sentences(cleaned)

    if not result["sentences"]:
        result["status"] = "failed"
        result["notes"] = "No valid sentences after segmentation"

    return result


print("Source processor defined ✓")

Source processor defined ✓


## Main pipeline — process all sources & write outputs

> **Runtime note:** This cell fetches 221 URLs sequentially with a 0.5 s delay between requests. Expect **15–30 minutes** depending on network speed and server responsiveness. Progress is printed after each source.


In [12]:
# ── MAIN PIPELINE ────────────────────────────────────────────

INPUT_CSV = "validated_dataset.csv"

df_sources = pd.read_csv(INPUT_CSV)
total_sources = len(df_sources)
log.info("Loaded %d sources from %s", total_sources, INPUT_CSV)
print(f"Loaded {total_sources} sources\n")

# ── Accumulators ──
all_rows = []  # sentence-level
qc_rows = []  # one per document
exclusion_rows = []
failure_rows = []

processed = 0
excluded = 0
failed = 0

for idx, src in df_sources.iterrows():
    doc_num = idx + 1
    print(f"[{doc_num}/{total_sources}] {src['title'][:75]}…", end="  ")

    result = process_one_source(src.to_dict())
    document_id = make_document_id(result["url"])

    # ── Excluded ──
    if result["status"] == "excluded":
        excluded += 1
        exclusion_rows.append(
            {
                "url": result["url"],
                "title": result["title"],
                "category": result["category"],
                "source_type": result["source_type"],
                "page_count": result["page_count"],
                "reason_for_exclusion": result["exclusion_reason"],
            }
        )
        qc_rows.append(
            {
                "document_id": document_id,
                "title": result["title"],
                "url": result["url"],
                "category": result["category"],
                "source_type": result["source_type"],
                "page_count": result["page_count"],
                "extraction_status": "excluded",
                "exclusion_status": "excluded",
                "total_sentences": 0,
                "equity_sentences": 0,
                "notes": result["exclusion_reason"],
            }
        )
        print(f"EXCLUDED — {result['exclusion_reason']}")
        continue

    # ── Failed ──
    if result["status"] in ("failed", "fetch_failed"):
        failed += 1
        failure_rows.append(
            {
                "url": result["url"],
                "title": result["title"],
                "category": result["category"],
                "source_type": result["source_type"],
                "error": result["notes"],
            }
        )
        qc_rows.append(
            {
                "document_id": document_id,
                "title": result["title"],
                "url": result["url"],
                "category": result["category"],
                "source_type": result["source_type"],
                "page_count": result["page_count"],
                "extraction_status": result["status"],
                "exclusion_status": "failed",
                "total_sentences": 0,
                "equity_sentences": 0,
                "notes": result["notes"],
            }
        )
        print(f"FAILED — {result['notes'][:80]}")
        continue

    # ── Success: build sentence rows ──
    processed += 1
    sentences = result["sentences"]
    subtype = classify_subtype(
        result["title"], result["category"], result["raw_text_sample"]
    )

    equity_count = 0
    for si, sent_text in enumerate(sentences):
        sentence_id = f"{document_id}_s{si:05d}"
        features = compute_features(sent_text)

        prev_sent = sentences[si - 1] if si > 0 else ""
        next_sent = sentences[si + 1] if si < len(sentences) - 1 else ""

        all_rows.append(
            {
                "document_id": document_id,
                "sentence_id": sentence_id,
                "sentence_index_within_document": si,
                "sentence_text": sent_text,
                "category": result["category"],
                "title": result["title"],
                "source_label": result["source_label"],
                "url": result["url"],
                "source_type": result["source_type"],
                "page_count": result["page_count"],
                "document_subtype": subtype,
                "prev_sentence": prev_sent,
                "next_sentence": next_sent,
                **features,
            }
        )
        if (
            features["contains_equity"]
            or features["contains_equities"]
            or features["contains_health_equity"]
        ):
            equity_count += 1

    qc_rows.append(
        {
            "document_id": document_id,
            "title": result["title"],
            "url": result["url"],
            "category": result["category"],
            "source_type": result["source_type"],
            "page_count": result["page_count"],
            "extraction_status": "success",
            "exclusion_status": "included",
            "total_sentences": len(sentences),
            "equity_sentences": equity_count,
            "notes": "",
        }
    )
    print(f"OK — {len(sentences)} sentences ({equity_count} equity)")

    time.sleep(0.5)

print(f"\n{'='*60}")
print(f"Processing complete.")
print(f"  Processed : {processed}")
print(f"  Excluded  : {excluded}")
print(f"  Failed    : {failed}")

2026-04-15 20:20:12,652  INFO      Loaded 221 sources from validated_dataset.csv
Loaded 221 sources

[1/221] Developing Health Equity Measures…  OK — 808 sentences (184 equity)
[2/221] HHS Action Plan to Reduce Racial and Ethnic Health Disparities…  OK — 518 sentences (21 equity)
[3/221] Agency Program, Activity, and Policy Highlights to Advance Health Equity an…  OK — 830 sentences (34 equity)
[4/221] Advancing Equity through Quantitative Analysis…  EXCLUDED — page_count=14 < 20
[5/221] 2023 Capacity Assessment Update…  OK — 272 sentences (5 equity)
[6/221] Landscape of Area-Level Deprivation Measures and Other Approaches to Accoun…  OK — 1597 sentences (8 equity)
[7/221] Building Data Capacity for Patient-Centered Outcomes Research to Address CO…  OK — 430 sentences (2 equity)
[8/221] Health Equity Guide…  EXCLUDED — page_count=7 < 20
[9/221] Focusing on Health Equity…  2026-04-15 20:20:58,927  WARNING   Attempt 1/2 failed for https://stacks.cdc.gov/view/cdc/126226/cdc_126226_DS1.pdf

KeyboardInterrupt: 

## Write all output CSVs (same folder)

In [ ]:
# ── Write outputs to the SAME folder ─────────────────────────

# 1. Full sentence corpus
df_full = pd.DataFrame(all_rows)
df_full.to_csv(
    OUTPUT_DIR / "full_sentence_corpus.csv", index=False, encoding="utf-8-sig"
)
print(f"✓ full_sentence_corpus.csv        — {len(df_full):,} rows")

# 2. Equity subset
if not df_full.empty:
    mask = (
        df_full["contains_equity"]
        | df_full["contains_equities"]
        | df_full["contains_health_equity"]
    )
    df_equity = df_full[mask].copy()
else:
    df_equity = pd.DataFrame()
df_equity.to_csv(
    OUTPUT_DIR / "equity_sentence_subset.csv", index=False, encoding="utf-8-sig"
)
print(f"✓ equity_sentence_subset.csv      — {len(df_equity):,} rows")

# 3. QC report
df_qc = pd.DataFrame(qc_rows)
df_qc.to_csv(OUTPUT_DIR / "extraction_qc_report.csv", index=False, encoding="utf-8-sig")
print(f"✓ extraction_qc_report.csv        — {len(df_qc):,} rows")

# 4. Exclusion log
df_excl = pd.DataFrame(exclusion_rows)
df_excl.to_csv(OUTPUT_DIR / "exclusion_log.csv", index=False, encoding="utf-8-sig")
print(f"✓ exclusion_log.csv               — {len(df_excl):,} rows")

# 5. Failure log
df_fail = pd.DataFrame(failure_rows)
df_fail.to_csv(OUTPUT_DIR / "failure_log.csv", index=False, encoding="utf-8-sig")
print(f"✓ failure_log.csv                 — {len(df_fail):,} rows")

✓ full_sentence_corpus.csv        — 27,365 rows
✓ equity_sentence_subset.csv      — 4,061 rows
✓ extraction_qc_report.csv        — 221 rows
✓ exclusion_log.csv               — 27 rows
✓ failure_log.csv                 — 66 rows


## Summary statistics

In [ ]:
# ── Summary ──────────────────────────────────────────────────

print("=" * 60)
print("CORPUS BUILD SUMMARY")
print("=" * 60)
print(f"Total sources:      {total_sources}")
print(f"Processed sources:  {processed}")
print(f"Excluded sources:   {excluded}")
print(f"Failed sources:     {failed}")
print("-" * 40)
print(f"Total sentences:    {len(df_full):,}")
print(f"Equity sentences:   {len(df_equity):,}")
print()

if not df_full.empty:
    print("Sentences per category:")
    for cat, cnt in df_full["category"].value_counts().items():
        print(f"  {cat:<28s} {cnt:,}")
    print()

if not df_equity.empty:
    print("Equity sentences per category:")
    for cat, cnt in df_equity["category"].value_counts().items():
        print(f"  {cat:<28s} {cnt:,}")
    print()

print("Unique documents:", df_full["document_id"].nunique() if not df_full.empty else 0)
print("=" * 60)

CORPUS BUILD SUMMARY
Total sources:      221
Processed sources:  128
Excluded sources:   27
Failed sources:     66
----------------------------------------
Total sentences:    27,365
Equity sentences:   4,061

Sentences per category:
  state_local                  12,337
  federal_policy               8,533
  ngo_nonprofit                5,581
  news_commentary              483
  academic                     431

Equity sentences per category:
  state_local                  2,203
  federal_policy               1,036
  ngo_nonprofit                630
  academic                     140
  news_commentary              52

Unique documents: 128


## Quick preview of the corpus

In [ ]:
# ── Preview ──────────────────────────────────────────────────
if not df_full.empty:
    print("── full_sentence_corpus.csv (first 5 rows) ──")
    display(
        df_full[
            [
                "document_id",
                "sentence_index_within_document",
                "sentence_text",
                "category",
                "contains_equity",
                "document_subtype",
            ]
        ].head()
    )

    print("\n── equity_sentence_subset.csv (first 5 rows) ──")
    display(
        df_equity[
            [
                "document_id",
                "sentence_text",
                "category",
                "contains_health_equity",
                "is_definition_sentence",
            ]
        ].head()
    )
else:
    print("No data — check failure_log.csv and exclusion_log.csv")

── full_sentence_corpus.csv (first 5 rows) ──


,document_id,sentence_index_within_document,sentence_text,category,contains_equity,document_subtype
0,doc_2d830432db58,0,eveloping Health Equity Measures Prepared for ...,federal_policy,True,policy_report
1,doc_2d830432db58,1,ASPE leads special initiatives; coordinates th...,federal_policy,False,policy_report
2,doc_2d830432db58,2,"Integral to this role, ASPE conducts research ...",federal_policy,False,policy_report
3,doc_2d830432db58,3,Office of Health Policy The Office of Health P...,federal_policy,False,policy_report
4,doc_2d830432db58,4,HP carries out this mission by conducting poli...,federal_policy,False,policy_report



── equity_sentence_subset.csv (first 5 rows) ──


,document_id,sentence_text,category,contains_health_equity,is_definition_sentence
0,doc_2d830432db58,eveloping Health Equity Measures Prepared for ...,federal_policy,True,False
8,doc_2d830432db58,This included the recommendations that the Cen...,federal_policy,True,False
9,doc_2d830432db58,"Moreover, in the ASPE commissioned report, Sys...",federal_policy,True,False
10,doc_2d830432db58,"In response to this challenge, ASPE asked the ...",federal_policy,True,False
11,doc_2d830432db58,RAND identified 10 existing approaches to heal...,federal_policy,True,False


## How `document_id` controls for sentence dependency

Sentences from the same document are **not independent** — they share topic, author style, and context. The `document_id` column lets you:

1. **Sample *k* sentences per document** to avoid any single long report dominating the corpus:
   ```python
   sampled = df_full.groupby("document_id").apply(
       lambda g: g.sample(min(len(g), 5), random_state=42)
   ).reset_index(drop=True)
   ```

2. **Use `document_id` as a clustering / random-effects variable** in mixed-effects models so that standard errors account for within-document correlation.

3. **Reconstruct reading order** via `sentence_index_within_document` if you need to examine how equity framing unfolds across a document.

## Debugging tips

| Symptom | Where to look | Fix |
|---|---|---|
| Source not in corpus | `exclusion_log.csv` | Page count outside 20–150; adjust thresholds if needed |
| Source shows 0 sentences | `extraction_qc_report.csv` | Open the URL manually; it may be a scan-only PDF |
| Fetch error (403/timeout) | `failure_log.csv` | Try downloading the PDF manually and adding a local-file path |
| PubMed abstract empty | `failure_log.csv` | PubMed may have changed HTML structure; update `_extract_pubmed_abstract()` selectors |
| Too many short fragments | Adjust `MIN_SENTENCE_CHARS` | Raise from 20 → 30 to be stricter |
